# Training a microWakeWord Model

This notebook steps you through training a basic microWakeWord model. It is intended as a **starting point** for advanced users. You should use Python 3.10.

**The model generated will most likely not be usable for everyday use; it may be difficult to trigger or falsely activates too frequently. You will most likely have to experiment with many different settings to obtain a decent model!**

In the comment at the start of certain blocks, I note some specific settings to consider modifying.

This runs on Google Colab, but is extremely slow compared to training on a local GPU. If you must use Colab, be sure to Change the runtime type to a GPU. Even then, it still slow!

At the end of this notebook, you will be able to download a tflite file. To use this in ESPHome, you need to write a model manifest JSON file. See the [ESPHome documentation](https://esphome.io/components/micro_wake_word) for the details and the [model repo](https://github.com/esphome/micro-wake-word-models/tree/main/models/v2) for examples.

In [1]:
# Installs microWakeWord. Restart the kernel after this cell finishes.
import platform
import sys
from pathlib import Path

print("Kernel Python:", sys.executable)

# `pymicro-features` must expose `process_samples` for feature generation.
!{sys.executable} -m pip uninstall -y pymicro-features micro-features micro_features_cpp
!{sys.executable} -m pip install --no-cache-dir --force-reinstall "pymicro-features==2.0.2"

if platform.system() == "Darwin":
    # Keep the macOS-friendly audio-metadata dependency from the working notes.
    !{sys.executable} -m pip install 'git+https://github.com/whatsnowplaying/audio-metadata@d4ebb238e6a401bb1a5aaaac60c9e2b3cb30929f'

repo_dir = Path("microWakeWord")
if not repo_dir.exists():
    !git clone https://github.com/kahrendt/microWakeWord

# Patch train.py for newer NumPy / TensorFlow return types.
train_py = repo_dir / "microwakeword" / "train.py"
if train_py.exists():
    text = train_py.read_text()
    if "def as_array(value):" not in text:
        text = text.replace(
            "def validate_nonstreaming(config, data_processor, model, test_set):\n",
            "def validate_nonstreaming(config, data_processor, model, test_set):\n    def as_array(value):\n        return value.numpy() if hasattr(value, \"numpy\") else value\n\n",
        )
    text = text.replace('test_set_fp = result["fp"].numpy()', 'test_set_fp = as_array(result["fp"])')
    text = text.replace('all_true_positives = ambient_predictions["tp"].numpy()', 'all_true_positives = as_array(ambient_predictions["tp"])')
    text = text.replace('ambient_false_positives = ambient_predictions["fp"].numpy() - test_set_fp', 'ambient_false_positives = as_array(ambient_predictions["fp"]) - test_set_fp')
    text = text.replace('all_false_negatives = ambient_predictions["fn"].numpy()', 'all_false_negatives = as_array(ambient_predictions["fn"])')
    text = text.replace('np.trapz(np.flip(y_coordinates), np.flip(x_coordinates)) / 2.0', 'np.trapezoid(np.flip(y_coordinates), np.flip(x_coordinates)) / 2.0')
    train_py.write_text(text)

# Patch test.py for newer NumPy versions.
test_py = repo_dir / "microwakeword" / "test.py"
if test_py.exists():
    text = test_py.read_text()
    text = text.replace('np.trapz(y_coordinates, x_coordinates)', 'np.trapezoid(y_coordinates, x_coordinates)')
    test_py.write_text(text)

!{sys.executable} -m pip install -e ./microWakeWord

from pymicro_features import MicroFrontend
frontend = MicroFrontend()
print("Has process_samples:", hasattr(frontend, "process_samples"))

Kernel Python: /opt/homebrew/opt/python@3.11/bin/python3.11
Found existing installation: pymicro-features 2.0.2
Uninstalling pymicro-features-2.0.2:
  Successfully uninstalled pymicro-features-2.0.2
  Cloning https://github.com/whatsnowplaying/audio-metadata (to revision d4ebb238e6a401bb1a5aaaac60c9e2b3cb30929f) to /private/var/folders/xy/4wf6v9v52pgdxs5_d640g7pr0000gn/T/pip-req-build-jrgu12s3
  Running command git clone --filter=blob:none --quiet https://github.com/whatsnowplaying/audio-metadata /private/var/folders/xy/4wf6v9v52pgdxs5_d640g7pr0000gn/T/pip-req-build-jrgu12s3
  Running command git rev-parse -q --verify 'sha^d4ebb238e6a401bb1a5aaaac60c9e2b3cb30929f'
  Running command git fetch -q https://github.com/whatsnowplaying/audio-metadata d4ebb238e6a401bb1a5aaaac60c9e2b3cb30929f
  Running command git checkout -q d4ebb238e6a401bb1a5aaaac60c9e2b3cb30929f
  Resolved https://github.com/whatsnowplaying/audio-metadata to commit d4ebb238e6a401bb1a5aaaac60c9e2b3cb30929f
  Installing build

In [2]:
# Generates 1 sample of the target word for manual verification.

target_word = 'hey friday'  # Phonetic spellings may produce better samples

import os
import platform
import sys
from pathlib import Path

from IPython.display import Audio

print("Kernel Python:", sys.executable)

sample_generator_dir = Path("piper-sample-generator")
model_path = sample_generator_dir / "models" / "en_US-libritts_r-medium.pt"

if not sample_generator_dir.exists():
    if platform.system() == "Darwin":
        !git clone -b mps-support https://github.com/kahrendt/piper-sample-generator
    else:
        !git clone https://github.com/rhasspy/piper-sample-generator

sample_generator_dir.joinpath("models").mkdir(parents=True, exist_ok=True)

# PyTorch 2.6+ changed torch.load to default to weights_only=True.
# Patch the generated dependency in-place so rerunning from a clean clone works.
generate_samples_py = sample_generator_dir / "generate_samples.py"
if generate_samples_py.exists():
    text = generate_samples_py.read_text()
    old = "torch_model = torch.load(model_path)"
    new = "torch_model = torch.load(model_path, weights_only=False)"
    if old in text and new not in text:
        generate_samples_py.write_text(text.replace(old, new))

if not model_path.exists():
    !curl -L 'https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt' -o '{model_path}'

# Install dependencies into the active notebook environment.
!{sys.executable} -m pip install torch torchaudio piper-phonemize-cross==1.2.1

if "piper-sample-generator/" not in sys.path:
    sys.path.append("piper-sample-generator/")

!{sys.executable} piper-sample-generator/generate_samples.py "{target_word}" \
--max-samples 1 \
--batch-size 1 \
--output-dir generated_samples

sample_path = Path("generated_samples/0.wav")
if not sample_path.exists():
    raise FileNotFoundError(f"Sample generation failed, missing: {sample_path}")

Audio(filename=str(sample_path), autoplay=True)

Kernel Python: /opt/homebrew/opt/python@3.11/bin/python3.11
Cloning into 'piper-sample-generator'...
remote: Enumerating objects: 135, done.
remote: Counting objects: 100% (69/69), done.
remote: Compressing objects: 100% (25/25), done.
remote: Total 135 (delta 53), reused 44 (delta 44), pack-reused 66 (from 1)
Receiving objects: 100% (135/135), 1.03 MiB | 2.21 MiB/s, done.
Resolving deltas: 100% (61/61), done.
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100  194M  100  194M    0     0  2766k      0  0:01:12  0:01:12 --:--:-- 1771k
/opt/homebrew/lib/python3.11/site-packages/webrtcvad.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this pa

In [3]:
# Generates a larger amount of wake word samples.
# Start here when trying to improve your model.
# See https://github.com/rhasspy/piper-sample-generator for the full set of
# parameters. In particular, experiment with noise-scales and noise-scale-ws,
# generating negative samples similar to the wake word, and generating many more
# wake word samples, possibly with different phonetic pronunciations.

!{sys.executable} piper-sample-generator/generate_samples.py "{target_word}" \
--max-samples 1000 \
--batch-size 100 \
--output-dir generated_samples

/opt/homebrew/lib/python3.11/site-packages/webrtcvad.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
DEBUG:__main__:Loading /Users/filippospirpilidis/Projects/wakewords-models/jupiter/micro-wake-word/notebooks/piper-sample-generator/models/en_US-libritts_r-medium.pt
INFO:__main__:Successfully loaded the model
DEBUG:__main__:MPS available, using GPU
/opt/homebrew/lib/python3.11/site-packages/torch/nn/functional.py:5430: UserWarning: MPS: The constant padding of more than 3 dimensions is not currently supported natively. It uses View Ops default implementation to run. This may have performance implications. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/aten/src/ATen/native/mps/operations/Pad.mm:476.)
  return torch._C._nn.pad(input, pad, mode, val

In [4]:
# Downloads audio data for augmentation. This can be slow!
# Borrowed from openWakeWord's automatic_model_training.ipynb, accessed March 4, 2024
#
# **Important note!** The data downloaded here has a mixture of difference
# licenses and usage restrictions. As such, any custom models trained with this
# data should be considered as appropriate for **non-commercial** personal use only.


import datasets
import scipy
import os
import shutil

import numpy as np

from pathlib import Path
from huggingface_hub import snapshot_download
from tqdm import tqdm

## Download MIR RIR data

output_dir = Path("./mit_rirs")
if not output_dir.exists():
    output_dir.mkdir()
    rir_repo = Path(snapshot_download(
        repo_id="davidscripka/MIT_environmental_impulse_responses",
        repo_type="dataset",
        allow_patterns=["16khz/*.wav"],
    ))
    for wav_path in tqdm(sorted((rir_repo / "16khz").glob("*.wav"))):
        shutil.copy2(wav_path, output_dir / wav_path.name)

## Download noise and background audio

# Audioset Dataset (https://research.google.com/audioset/dataset/index.html)
# Download one part of the audioset .tar files, extract, and convert to 16khz
# For full-scale training, it's recommended to download the entire dataset from
# https://huggingface.co/datasets/agkphysics/AudioSet, and
# even potentially combine it with other background noise datasets (e.g., FSD50k, Freesound, etc.)

if not os.path.exists("audioset"):
    os.mkdir("audioset")

    fname = "bal_train09.tar"
    out_dir = f"audioset/{fname}"
    link = "https://huggingface.co/datasets/agkphysics/AudioSet/resolve/main/data/" + fname
    !curl -L "{link}" -o "{out_dir}"
    !tar -xf "{out_dir}" -C audioset

    output_dir = "./audioset_16k"
    if not os.path.exists(output_dir):
        os.mkdir(output_dir)

    # Save clips to 16-bit PCM wav files
    audioset_dataset = datasets.Dataset.from_dict({"audio": [str(i) for i in Path("audioset/audio").glob("**/*.flac")]})
    audioset_dataset = audioset_dataset.cast_column("audio", datasets.Audio(sampling_rate=16000))
    for row in tqdm(audioset_dataset):
        name = row['audio']['path'].split('/')[-1].replace(".flac", ".wav")
        scipy.io.wavfile.write(os.path.join(output_dir, name), 16000, (row['audio']['array']*32767).astype(np.int16))

# Free Music Archive dataset
# https://github.com/mdeff/fma
# (Third-party mchl914 extra small set)

output_dir = "./fma"
if not os.path.exists(output_dir):
    os.mkdir(output_dir)
    fname = "fma_xs.zip"
    link = "https://huggingface.co/datasets/mchl914/fma_xsmall/resolve/main/" + fname
    out_dir = f"fma/{fname}"
    !curl -L "{link}" -o "{out_dir}"
    !unzip -q "{out_dir}" -d "{output_dir}"

    output_dir = "./fma_16k"
    if not os.path.exists(output_dir):
        os.mkdir(output_dir)

    # Save clips to 16-bit PCM wav files
    fma_dataset = datasets.Dataset.from_dict({"audio": [str(i) for i in Path("fma/fma_small").glob("**/*.mp3")]})
    fma_dataset = fma_dataset.cast_column("audio", datasets.Audio(sampling_rate=16000))
    for row in tqdm(fma_dataset):
        name = row['audio']['path'].split('/')[-1].replace(".mp3", ".wav")
        scipy.io.wavfile.write(os.path.join(output_dir, name), 16000, (row['audio']['array']*32767).astype(np.int16))


Fetching 270 files:   0%|          | 0/270 [00:00<?, ?it/s]


100%|███████████████████████████████████████████████████| 270/270 [00:00<00:00, 2050.17it/s]

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0

100    15  100    15    0     0     42      0 --:--:-- --:--:-- --:--:--    42
tar: Error opening archive: Unrecognized archive format



0it [00:00, ?it/s]

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0

100  1375  100  1375    0     0   3760      0 --:--:-- --:--:-- --:--:--  3767
100  173M  100  173M    0     0  17.3M      0  0:00:10  0:00:10 --:--:-- 18.9M



100%|█████████████████████████████████████████████████████| 210/210 [00:09<00:00, 22.24it/s]


In [5]:
# Sets up the augmentations.
# To improve your model, experiment with these settings and use more sources of
# background clips.

import audiomentations

if not hasattr(audiomentations, "AddColorNoise"):
    print("audiomentations.AddColorNoise is unavailable; using a no-op fallback.")

    def _add_color_noise_fallback(*args, **kwargs):
        p = kwargs.get("p", 0.0)
        return audiomentations.Lambda(transform=lambda samples, sample_rate: samples, p=p)

    audiomentations.AddColorNoise = _add_color_noise_fallback

from microwakeword.audio.augmentation import Augmentation
from microwakeword.audio.clips import Clips
from microwakeword.audio.spectrograms import SpectrogramGeneration

clips = Clips(input_directory='generated_samples',
              file_pattern='*.wav',
              max_clip_duration_s=None,
              remove_silence=False,
              random_split_seed=10,
              split_count=0.1,
              )
augmenter = Augmentation(augmentation_duration_s=3.2,
                         augmentation_probabilities = {
                                "SevenBandParametricEQ": 0.1,
                                "TanhDistortion": 0.1,
                                "PitchShift": 0.1,
                                "BandStopFilter": 0.1,
                                "AddColorNoise": 0.1,
                                "AddBackgroundNoise": 0.75,
                                "Gain": 1.0,
                                "RIR": 0.5,
                            },
                         impulse_paths = ['mit_rirs'],
                         background_paths = ['fma_16k', 'audioset_16k'],
                         background_min_snr_db = -5,
                         background_max_snr_db = 10,
                         min_jitter_s = 0.195,
                         max_jitter_s = 0.205,
                         )


audiomentations.AddColorNoise is unavailable; using a no-op fallback.


    pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.


In [6]:
# Augment a random clip and play it back to verify it works well.
# Normalize only for listening so quiet clips are still audible in Jupyter.

import numpy as np
from IPython.display import Audio
from microwakeword.audio.audio_utils import save_clip

random_clip = clips.get_random_clip()
augmented_clip = augmenter.augment_clip(random_clip)

peak = float(np.max(np.abs(augmented_clip))) if len(augmented_clip) else 0.0
print("Augmented clip peak:", peak)

preview_clip = augmented_clip
if peak > 0:
    preview_clip = (augmented_clip / peak) * 0.95

save_clip(preview_clip, 'augmented_clip.wav')

Audio("augmented_clip.wav", autoplay=True)

Augmented clip peak: 0.07274025678634644


In [7]:
# Augment samples and save the training, validation, and testing sets.
# Validating and testing samples generated the same way can make the model
# benchmark better than it performs in real-word use. Use real samples or TTS
# samples generated with a different TTS engine to potentially get more accurate
# benchmarks.

import os
from mmap_ninja.ragged import RaggedMmap

output_dir = 'generated_augmented_features'

if not os.path.exists(output_dir):
    os.mkdir(output_dir)

splits = ["training", "validation", "testing"]
for split in splits:
  out_dir = os.path.join(output_dir, split)
  if not os.path.exists(out_dir):
      os.mkdir(out_dir)


  split_name = "train"
  repetition = 2

  spectrograms = SpectrogramGeneration(clips=clips,
                                     augmenter=augmenter,
                                     slide_frames=10,    # Uses the same spectrogram repeatedly, just shifted over by one frame. This simulates the streaming inferences while training/validating in nonstreaming mode.
                                     step_ms=10,
                                     )
  if split == "validation":
    split_name = "validation"
    repetition = 1
  elif split == "testing":
    split_name = "test"
    repetition = 1
    spectrograms = SpectrogramGeneration(clips=clips,
                                     augmenter=augmenter,
                                     slide_frames=1,    # The testing set uses the streaming version of the model, so no artificial repetition is necessary
                                     step_ms=10,
                                     )

  RaggedMmap.from_generator(
      out_dir=os.path.join(out_dir, 'wakeword_mmap'),
      sample_generator=spectrograms.spectrogram_generator(split=split_name, repeat=repetition),
      batch_size=100,
      verbose=True,
  )

0it [00:00, ?it/s]

0it [00:00, ?it/s]

0it [00:00, ?it/s]

In [8]:
# Downloads pre-generated spectrogram features (made for microWakeWord in
# particular) for various negative datasets. This can be slow!

output_dir = './negative_datasets'
if not os.path.exists(output_dir):
    os.mkdir(output_dir)
    link_root = "https://huggingface.co/datasets/kahrendt/microwakeword/resolve/main/"
    filenames = ['dinner_party.zip', 'dinner_party_eval.zip', 'no_speech.zip', 'speech.zip']
    for fname in filenames:
        link = link_root + fname

        zip_path = f"negative_datasets/{fname}"
        !curl -L "{link}" -o "{zip_path}"
        !unzip -q "{zip_path}" -d "{output_dir}"

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  1399  100  1399    0     0   3446      0 --:--:-- --:--:-- --:--:--  3445
100  423M  100  423M    0     0  14.8M      0  0:00:28  0:00:28 --:--:-- 16.8M
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  1399  100  1399    0     0   5718      0 --:--:-- --:--:-- --:--:--  5733
100 78.5M  100 78.5M    0     0  20.9M      0  0:00:03  0:00:03 --:--:-- 25.3M
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  1379  100  1379    0     0   2290      0 --:--:-- --:--:-- --:--:--  2294
100 1907M  100 1907M    0     0  21.1M      0  0:01:30  0:01:30 --:--:-- 17.6M
  % Total    % Received % Xferd  Average Speed   Tim

In [9]:
# Save a yaml config that controls the training process
# These hyperparamters can make a huge different in model quality.
# Experiment with sampling and penalty weights and increasing the number of
# training steps.

import yaml
import os

config = {}

config["window_step_ms"] = 10

config["train_dir"] = (
    "trained_models/wakeword"
)


# Each feature_dir should have at least one of the following folders with this structure:
#  training/
#    ragged_mmap_folders_ending_in_mmap
#  testing/
#    ragged_mmap_folders_ending_in_mmap
#  testing_ambient/
#    ragged_mmap_folders_ending_in_mmap
#  validation/
#    ragged_mmap_folders_ending_in_mmap
#  validation_ambient/
#    ragged_mmap_folders_ending_in_mmap
#
#  sampling_weight: Weight for choosing a spectrogram from this set in the batch
#  penalty_weight: Penalizing weight for incorrect predictions from this set
#  truth: Boolean whether this set has positive samples or negative samples
#  truncation_strategy = If spectrograms in the set are longer than necessary for training, how are they truncated
#       - random: choose a random portion of the entire spectrogram - useful for long negative samples
#       - truncate_start: remove the start of the spectrogram
#       - truncate_end: remove the end of the spectrogram
#       - split: Split the longer spectrogram into separate spectrograms offset by 100 ms. Only for ambient sets

config["features"] = [
    {
        "features_dir": "generated_augmented_features",
        "sampling_weight": 2.0,
        "penalty_weight": 1.0,
        "truth": True,
        "truncation_strategy": "truncate_start",
        "type": "mmap",
    },
    {
        "features_dir": "negative_datasets/speech",
        "sampling_weight": 10.0,
        "penalty_weight": 1.0,
        "truth": False,
        "truncation_strategy": "random",
        "type": "mmap",
    },
    {
        "features_dir": "negative_datasets/dinner_party",
        "sampling_weight": 10.0,
        "penalty_weight": 1.0,
        "truth": False,
        "truncation_strategy": "random",
        "type": "mmap",
    },
    {
        "features_dir": "negative_datasets/no_speech",
        "sampling_weight": 5.0,
        "penalty_weight": 1.0,
        "truth": False,
        "truncation_strategy": "random",
        "type": "mmap",
    },
    { # Only used for validation and testing
        "features_dir": "negative_datasets/dinner_party_eval",
        "sampling_weight": 0.0,
        "penalty_weight": 1.0,
        "truth": False,
        "truncation_strategy": "split",
        "type": "mmap",
    },
]

# Number of training steps in each iteration - various other settings are configured as lists that corresponds to different steps
config["training_steps"] = [10000]

# Penalizing weight for incorrect class predictions - lists that correspond to training steps
config["positive_class_weight"] = [1]
config["negative_class_weight"] = [20]

config["learning_rates"] = [
    0.001,
]  # Learning rates for Adam optimizer - list that corresponds to training steps
config["batch_size"] = 128

config["time_mask_max_size"] = [
    0
]  # SpecAugment - list that corresponds to training steps
config["time_mask_count"] = [0]  # SpecAugment - list that corresponds to training steps
config["freq_mask_max_size"] = [
    0
]  # SpecAugment - list that corresponds to training steps
config["freq_mask_count"] = [0]  # SpecAugment - list that corresponds to training steps

config["eval_step_interval"] = (
    500  # Test the validation sets after every this many steps
)
config["clip_duration_ms"] = (
    1500  # Maximum length of wake word that the streaming model will accept
)

# The best model weights are chosen first by minimizing the specified minimization metric below the specified target_minimization
# Once the target has been met, it chooses the maximum of the maximization metric. Set 'minimization_metric' to None to only maximize
# Available metrics:
#   - "loss" - cross entropy error on validation set
#   - "accuracy" - accuracy of validation set
#   - "recall" - recall of validation set
#   - "precision" - precision of validation set
#   - "false_positive_rate" - false positive rate of validation set
#   - "false_negative_rate" - false negative rate of validation set
#   - "ambient_false_positives" - count of false positives from the split validation_ambient set
#   - "ambient_false_positives_per_hour" - estimated number of false positives per hour on the split validation_ambient set
config["target_minimization"] = 0.9
config["minimization_metric"] = None  # Set to None to disable

config["maximization_metric"] = "average_viable_recall"

with open(os.path.join("training_parameters.yaml"), "w") as file:
    documents = yaml.dump(config, file)

In [10]:
# Trains a model. When finished, it will quantize and convert the model to a
# streaming version suitable for on-device detection.
# It will resume if stopped, but it will start over at the configured training
# steps in the yaml file.
# Change --train 0 to only convert and test the best-weighted model.
# On Google colab, it doesn't print the mini-batch results, so it may appear
# stuck for several minutes! Additionally, it is very slow compared to training
# on a local GPU.

import sys

print("Kernel Python:", sys.executable)

!{sys.executable} -m microwakeword.model_train_eval \
--training_config='training_parameters.yaml' \
--train 1 \
--restore_checkpoint 1 \
--test_tf_nonstreaming 0 \
--test_tflite_nonstreaming 0 \
--test_tflite_nonstreaming_quantized 0 \
--test_tflite_streaming 0 \
--test_tflite_streaming_quantized 1 \
--use_weights "best_weights" \
mixednet \
--pointwise_filters "64,64,64,64" \
--repeat_in_block  "1, 1, 1, 1" \
--mixconv_kernel_sizes '[5], [7,11], [9,15], [23]' \
--residual_connection "0,0,0,0" \
--first_conv_filters 32 \
--first_conv_kernel_size 5 \
--stride 3

Kernel Python: /opt/homebrew/opt/python@3.11/bin/python3.11
    pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
INFO:absl:Loading and analyzing data sets.
Model: "functional"
┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (128, 204, 40)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expand_dims         │ (128, 204, 1, 40) │          0 │ input_layer[0][0] │
│ (ExpandDims)        │                   │            │                   │
├─────────────────────┼─────

In [11]:
# Shows the exported tflite model path for local Jupyter use.
# After training, copy this file to the wakewords-models repo as hey_mona.tflite.

from pathlib import Path

model_path = Path("trained_models/wakeword/tflite_stream_state_internal_quant/stream_state_internal_quant.tflite")
print("Exported model path:", model_path.resolve())
print("Model exists:", model_path.exists())
if not model_path.exists():
    raise FileNotFoundError(f"Model not found yet: {model_path}")

Exported model path: /Users/filippospirpilidis/Projects/wakewords-models/jupiter/micro-wake-word/notebooks/trained_models/wakeword/tflite_stream_state_internal_quant/stream_state_internal_quant.tflite
Model exists: True
